## ADIP Phase 2: Data Standardization Investigation

### Objective
Investigate raw Parquet data from Phase 1 ingestion to identify schema inconsistencies, missing values, type mismatches, and structural anomalies before enforcing `StandardizedRecord` validation.

### Investigation Targets
- **API ingestion output**: `api_auth*.csv`, `api_ingest*.csv`
- **Scraper output**: `scraper*.csv`

### Key Questions
1. Do all files share identical column names and types?
2. Are timestamps consistently formatted (UTC, timezone-aware)?
3. What percentage of records fail `StandardizedRecord` validation?
4. Which fields require coercion (string → float, object → datetime)?
5. What edge cases exist (empty strings, NaN patterns, outliers)?

### BEGIN!

  **FRMAEWORK FOR PHASE 1 - API_INGEST.CSV** 
--------------
#### 1. Phase Overview
In this phase, we ingest the raw Ecommerce Sales dataset and conduct a high-level inspection.
Goals:

- Load the dataset in a reproducible way.

- Examine basic metadata and structural integrity.

- Identify potential data quality issues for later cleaning.

- Document first impressions and possible red flags.

In [1]:
%pip install pandas
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

In [3]:
# Check the current working directory
import os
os.getcwd()

'c:\\Users\\CHARLES\\Desktop\\ADIP-Intelligence-lab\\eda_notebook'

In [34]:
df = pd.read_csv("../.data_cache/api_ingest.csv")  

#### Basic Structural Inspection
Checklist:

- Shape (.shape)

- Column names (.columns)

- Data info: data types & nullcouns  (.info)

- Sample rows (.head() / .tail())

In [5]:
df.head()

,name,brand,price,deal_price,final_price,description,image_thumbnail,sku,seller,stock,product_rating,fetched_at
0,JCO4 LAPTOP BATTERY,HP,24725,NaN,NaN,[object Object],/T/I/118566_1604505433.jpg,5000867,{'name': 'Konga'},"{'in_stock': True, 'quantity': 1}","{'quality': {'average': 0, 'number_of_ratings'...",2025-11-11 19:49:08
1,Basilisk Gaming Mouse Classic Black,Razer,244000,NaN,NaN,[object Object],/R/U/57289_1727279653.jpg,6566278,{'name': 'YOMILINCON BRAND'},"{'in_stock': True, 'quantity': 20}","{'quality': {'average': 0, 'number_of_ratings'...",2025-11-11 19:49:08
2,Ht03xl In-built Battery,HP,35075,NaN,NaN,[object Object],/S/H/118566_1618241664.jpg,5228829,{'name': 'Konga'},"{'in_stock': True, 'quantity': 2}","{'quality': {'average': 0, 'number_of_ratings'...",2025-11-11 19:49:08
3,HP Big-Mouth Laptop Charger - 18.5V 3.5A,NaN,10000,NaN,NaN,[object Object],/H/P/HP-Big-Mouth-Laptop-Charger---18-5V-3-5A-...,2353031,{'name': 'YOMILINCON BRAND'},"{'in_stock': True, 'quantity': 9}","{'quality': {'average': 4.5, 'number_of_rating...",2025-11-11 19:49:08
4,Baseus Ultrajoy 5w1 5-port Hub docking station...,NaN,49000,NaN,NaN,[object Object],/F/Z/57289_1760001914.jpg,6861874,{'name': 'YOMILINCON BRAND'},"{'in_stock': True, 'quantity': 2}","{'quality': {'average': 0, 'number_of_ratings'...",2025-11-11 19:49:08


In [6]:
print(f"shape :{df.shape}") 

shape :(1400, 12)


**Dropping deal_price, final_price & description columns - These are ghost columns with 0 data points.**

In [35]:
df.drop(columns=['deal_price', 'final_price', 'description'], inplace=True)
df.columns

Index(['name', 'brand', 'price', 'image_thumbnail', 'sku', 'seller', 'stock',
       'product_rating', 'fetched_at'],
      dtype='str')

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1400 entries, 0 to 1399
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   name             1400 non-null   str  
 1   brand            619 non-null    str  
 2   price            1400 non-null   int64
 3   image_thumbnail  1400 non-null   str  
 4   sku              1400 non-null   int64
 5   seller           1400 non-null   str  
 6   stock            1400 non-null   str  
 7   product_rating   1360 non-null   str  
 8   fetched_at       1400 non-null   str  
dtypes: int64(2), str(7)
memory usage: 98.6 KB


#### Inspection For Missing & Null Value  
Checklist:

- Count missing values per column (.isnull().sum()).

- Percentage of missing values.

- Repopulate missing data (e.g., 'NA', '?', 'n/a').



In [37]:
df.isna().sum()

name                 0
brand              781
price                0
image_thumbnail      0
sku                  0
seller               0
stock                0
product_rating      40
fetched_at           0
dtype: int64

In [38]:
percent_missing = df.isna().sum() * 100 / len(df)
print(percent_missing)

name                0.000000
brand              55.785714
price               0.000000
image_thumbnail     0.000000
sku                 0.000000
seller              0.000000
stock               0.000000
product_rating      2.857143
fetched_at          0.000000
dtype: float64


***Upon observation - Over half of the brand rows consist of missing value. This needs to be addressed!***

In [39]:
# Extract the brands from the names column by means of split function, and assign it to the empty brand column. 
names = df["name"].str.split().str[0]
if df["brand"].isna().any() or (df["brand"] == "").any():
    df["brand"] = names 

In [40]:
df["brand"].isna().sum() 

np.int64(0)

In [41]:
df["brand"].value_counts() 

brand
Baseus      66
Hp          58
Apple       52
Mx          47
Wireless    45
            ..
Rog          1
Air          1
K95          1
Metal        1
Ac           1
Name: count, Length: 153, dtype: int64

In [42]:
# Data cleaning: Replace the incorrect brand name "JCO4" with the correct brand name "HP".

if (df["brand"] == "JCO4").any():
    df["brand"] = df["brand"].replace("JCO4", "Hp")
df["brand"].value_counts() 

brand
Hp          93
Baseus      66
Apple       52
Mx          47
Wireless    45
            ..
Rog          1
Air          1
K95          1
Metal        1
Ac           1
Name: count, Length: 152, dtype: int64

In [59]:
df["brand"] = df["brand"].astype(str)
df.dtypes  

product_name                       str
brand                              str
price                            int64
image_thumbnail                    str
product_id                       int64
stock_qty                        int64
seller_name                        str
product_rating                 float64
rating_count                   float64
fetched_time       datetime64[us, UTC]
source                             str
dtype: object

In [61]:
# Data cleaning: Replace the incorrect brand name "HP" with the correct brand name "Hp".
df["brand"] = df["brand"].replace("HP", "Hp")
df["brand"].value_counts()     

brand
Hp          95
Baseus      66
Apple       52
Mx          47
Wireless    45
            ..
Rog          1
Air          1
K95          1
Metal        1
Ac           1
Name: count, Length: 151, dtype: int64

***Unpacking the nested strings stored within the following columns - seller, stock & product_rating*** 

In [24]:
import ast

In [43]:
def unpack_dict(val, target_key, default_val=None):
    """
    Safely turns a string like "{'in_stock': True, 'quantity': 20}" into a dictionary,
    and extracts the target_key.
    """
    if pd.isna(val):
        return default_val
        
    try:
        # Translate string -> dict
        parsed_dict = ast.literal_eval(str(val))
        
        # Grab the key we want safely
        return parsed_dict.get(target_key, default_val)
        
    except (ValueError, SyntaxError):
        # If the string is broken, don't crash the notebook, just return the default
        return default_val

print("✅ Unpacker tool loaded into memory.")

✅ Unpacker tool loaded into memory.


In [44]:
# 1. Extract the quantity from the stock column
df['stock_qty'] = df['stock'].apply(lambda x: unpack_dict(x, target_key='quantity', default_val=0))

# 2. Extract the name from the seller column
df['seller_name'] = df['seller'].apply(lambda x: unpack_dict(x, target_key='name', default_val='Unknown'))

# 3. Extract the average quality rating from the product_rating column
df["product_rating"] = df["product_rating"].apply(lambda x: ast.literal_eval(str(x)) if pd.notna(x) else {})
df["rating_avg"] = df["product_rating"].apply(lambda d: d.get('quality', {}).get("average", np.nan)) 
df["rating_count"] = df["product_rating"].apply(lambda d: d.get('quality', {}).get("number_of_ratings", np.nan)) 
 

In [45]:
# converting the fetched_at column to datetime format and assign it to a new column called fetched_time.
df["fetched_time"] = pd.to_datetime(df["fetched_at"], utc=True)
df["fetched_time"].dtype

datetime64[us, UTC]

In [46]:
df.columns

Index(['name', 'brand', 'price', 'image_thumbnail', 'sku', 'seller', 'stock',
       'product_rating', 'fetched_at', 'stock_qty', 'seller_name',
       'rating_avg', 'rating_count', 'fetched_time'],
      dtype='str')

In [47]:
# Drop the original stock, seller, fetched_at and product_rating columns
df.drop(columns=['stock', 'seller', 'product_rating', 'fetched_at'], inplace=True)

In [ ]:
# Final cleanup: rename the columns to match our desired schema, and reorder the columns.
df['source'] = "api_ingest"
df.rename(columns={"sku": "product_id", "name": "product_name",'rating_avg': 'product_rating'  }, inplace=True)
df = df.reindex(columns=['product_id', 'product_name', 'brand', 'price', 'product_rating', 'rating_count', 'seller_name', 'stock_qty', 'category', 'fetched_time', 'source' ]) 

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,category,fetched_time,source
0,5000867,JCO4 LAPTOP BATTERY,Hp,24725,0.0,0.0,Konga,1,NaN,2025-11-11 19:49:08+00:00,api_ingest
1,6566278,Basilisk Gaming Mouse Classic Black,Basilisk,244000,0.0,0.0,YOMILINCON BRAND,20,NaN,2025-11-11 19:49:08+00:00,api_ingest
2,5228829,Ht03xl In-built Battery,Ht03xl,35075,0.0,0.0,Konga,2,NaN,2025-11-11 19:49:08+00:00,api_ingest
3,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,HP,10000,4.5,2.0,YOMILINCON BRAND,9,NaN,2025-11-11 19:49:08+00:00,api_ingest
4,6861874,Baseus Ultrajoy 5w1 5-port Hub docking station...,Baseus,49000,0.0,0.0,YOMILINCON BRAND,2,NaN,2025-11-11 19:49:08+00:00,api_ingest
...,...,...,...,...,...,...,...,...,...,...,...
1395,6967053,Ac Adapter Laptop Charger For Lenovo Legion 5 ...,Ac,52000,0.0,0.0,Highspeed,20,NaN,2026-04-16 02:32:00+00:00,api_ingest
1396,6967018,Pd 100w 18.5-20v 7.4×0.6 To Usb-c For Dell Lap...,Pd,9000,0.0,0.0,Add Moore,20,NaN,2026-04-16 02:32:00+00:00,api_ingest
1397,6966990,Pd100w Usb-c 18.5-20v Small Blue Pin Cable For...,Pd100w,8000,0.0,0.0,Add Moore,20,NaN,2026-04-16 02:32:00+00:00,api_ingest
1398,6966926,Portable Relife XA5 Inductance Tester,Portable,5999,0.0,0.0,Justified Store,20,NaN,2026-04-16 02:32:00+00:00,api_ingest


In [65]:
# 4. VISUALIZE THE RESULTS
df = df.reindex(columns=['product_id', 'product_name', 'brand', 'price', 'product_rating', 'rating_count', 'seller_name', 'stock_qty', 'category', 'fetched_time', 'source' ]) 
df.head() 

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,category,fetched_time,source
0,5000867,JCO4 LAPTOP BATTERY,Hp,24725,0.0,0.0,Konga,1,NaN,2025-11-11 19:49:08+00:00,api_ingest
1,6566278,Basilisk Gaming Mouse Classic Black,Basilisk,244000,0.0,0.0,YOMILINCON BRAND,20,NaN,2025-11-11 19:49:08+00:00,api_ingest
2,5228829,Ht03xl In-built Battery,Ht03xl,35075,0.0,0.0,Konga,2,NaN,2025-11-11 19:49:08+00:00,api_ingest
3,2353031,HP Big-Mouth Laptop Charger - 18.5V 3.5A,Hp,10000,4.5,2.0,YOMILINCON BRAND,9,NaN,2025-11-11 19:49:08+00:00,api_ingest
4,6861874,Baseus Ultrajoy 5w1 5-port Hub docking station...,Baseus,49000,0.0,0.0,YOMILINCON BRAND,2,NaN,2025-11-11 19:49:08+00:00,api_ingest


  **FRMAEWORK FOR PHASE 2 - SCRAPER.CSV** 
--------------
#### Phase Overview
In this phase, we ingest the raw Ecommerce Sales dataset and conduct a high-level inspection.
Goals:

- Load the dataset in a reproducible way.

- Examine basic metadata and structural integrity.

- Identify potential data quality issues for later cleaning.

- Document first impressions and possible red flags.

In [21]:
df_2 = pd.read_csv("../.data_cache/scraper.csv")
df_2.head()

,Name,Price,Ratings,Description Link,Category,Timestamp
0,"Blueing 15.6"" Laptop J4125 8GB+256GB SSD Stude...","₦ 234,000",3.9 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13
1,Macbook PRO Laptop A1278 13.3 Inch Core I5 2.5...,"₦ 187,000",3.4 out of 5,https://www.jumia.com.ng/renewed-macbook-pro-l...,laptops,2025-11-13 01:35:13
2,"Blueing 14"" Laptop N3350 6GB+192GB SSD Portabl...","₦ 215,278",4.1 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13
3,Hp EliteBook 840 G7 Intel Core I5 Touchscreen ...,"₦ 566,000",4 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13
4,Hp Hp Hp EliteBook 840 G7 10th Gen Intel Core ...,"₦ 519,990",4.1 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13


In [22]:
df_2.info()

<class 'pandas.DataFrame'>
RangeIndex: 166993 entries, 0 to 166992
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Name              166993 non-null  str  
 1   Price             166993 non-null  str  
 2   Ratings           166993 non-null  str  
 3   Description Link  166993 non-null  str  
 4   Category          166993 non-null  str  
 5   Timestamp         166993 non-null  str  
dtypes: str(6)
memory usage: 7.6 MB


#### DATA STANDARDIZATION WORKFLOW -  
1. Create the Brand column

2. convert to appropriate data types

3. Standardize the Price column

4. Normalize the Rating column

**CREATING BRAND COLUMN**

In [23]:
df_2["brand"] = df_2["Name"].str.split().str[0]
df_2["brand"].value_counts()

brand
Hp         52819
DELL       19706
Samsung    18317
Nokia      11088
Apple      10247
           ...  
Item           1
Fitbit         1
SJBOB          1
K9             1
Visita         1
Name: count, Length: 137, dtype: int64

In [24]:
# Zero missing values in the dataset, which is a good sign for data quality.
df_2.isna().sum()

Name                0
Price               0
Ratings             0
Description Link    0
Category            0
Timestamp           0
brand               0
dtype: int64

**CONVERSION OF DATA TYPES & STANDARDIZATION**  

In [25]:
df_2["fetched_time"] = pd.to_datetime(df_2["Timestamp"], utc=True)
df_2["fetched_time"].dtype

datetime64[us, UTC]

In [26]:
# Standardizing the ratings column by extracting the first numeric part  
df_2["product_rating"] = df_2["Ratings"].str.split().str[0]
df_2["product_rating"] = pd.to_numeric(df_2["product_rating"], errors='coerce')

In [27]:
df_2["product_rating"].value_counts()

product_rating
5.0    20182
4.0    12913
4.5     6267
4.3     5601
3.0     5403
4.1     4729
3.8     4681
4.2     4564
3.9     3856
3.7     3528
3.5     3108
1.0     3062
4.4     3031
4.7     2261
3.6     2135
4.8     1876
3.3     1841
4.6     1753
3.4     1307
2.0     1030
2.5      676
3.2      653
2.7      354
4.9      343
3.1      324
2.3      225
2.8      180
1.5      120
1.3       23
1.8        3
Name: count, dtype: int64

In [28]:
# Cleaning the price column by removing the currency symbol and commas, then converting to numeric & renaming the Name, Category columns to product_name, category to match our desired schema. 
df_2["price"] = pd.to_numeric(df_2["Price"].str.replace("₦", "", regex=False).str.replace(",", "", regex=False), errors='coerce')
df_2.rename(columns={"Name": "product_name", "Category" : "category"}, inplace=True)

In [30]:
# Since the web scraped dataset doesn't have the rating_count, seller_name and stock_qty columns, 
# create them and fill them with NaN values to maintain consistency with our desired schema. 
# Adding a source column to indicate that this data came from web scraping.         
df_2["rating_count"] = np.nan
df_2["stock_qty"] = np.nan
df_2["seller_name"] = "JUMIA"
df_2["source"] = "web_scrape"
df_2.head()

,product_name,Price,Ratings,Description Link,category,Timestamp,brand,fetched_time,product_rating,price,rating_count,stock_qty,seller_name,source
0,"Blueing 15.6"" Laptop J4125 8GB+256GB SSD Stude...","₦ 234,000",3.9 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13,Blueing,2025-11-13 01:35:13+00:00,3.9,234000.0,NaN,NaN,JUMIA,web_scrape
1,Macbook PRO Laptop A1278 13.3 Inch Core I5 2.5...,"₦ 187,000",3.4 out of 5,https://www.jumia.com.ng/renewed-macbook-pro-l...,laptops,2025-11-13 01:35:13,Macbook,2025-11-13 01:35:13+00:00,3.4,187000.0,NaN,NaN,JUMIA,web_scrape
2,"Blueing 14"" Laptop N3350 6GB+192GB SSD Portabl...","₦ 215,278",4.1 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13,Blueing,2025-11-13 01:35:13+00:00,4.1,215278.0,NaN,NaN,JUMIA,web_scrape
3,Hp EliteBook 840 G7 Intel Core I5 Touchscreen ...,"₦ 566,000",4 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13,Hp,2025-11-13 01:35:13+00:00,4.0,566000.0,NaN,NaN,JUMIA,web_scrape
4,Hp Hp Hp EliteBook 840 G7 10th Gen Intel Core ...,"₦ 519,990",4.1 out of 5,https://www.jumia.com.ng/customer/account/logi...,laptops,2025-11-13 01:35:13,Hp,2025-11-13 01:35:13+00:00,4.1,519990.0,NaN,NaN,JUMIA,web_scrape


In [31]:
# Normalize the data present within the name column by generating a unique product_id using the MD5 hash of the product name. 
# This ensures that each product has a consistent identifier, even if there are variations in the name formatting.
import hashlib
def generate_product_id(name):
    if pd.isna(name):
        return None
    normalized = "".join(str(name).strip().lower().split())  
    return hashlib.md5(normalized.encode()).hexdigest()

df_2["product_id"] = df_2["product_name"].apply(generate_product_id) 

In [32]:
df_2 = df_2.reindex(columns=['product_id', 'product_name', 'brand', 'price', 'product_rating', 'rating_count', 'seller_name', 'stock_qty', 'category', 'fetched_time', 'source' ])
df_2.head()

,product_id,product_name,brand,price,product_rating,rating_count,seller_name,stock_qty,category,fetched_time,source
0,893da6d61fa159d1667b10601ae2a119,"Blueing 15.6"" Laptop J4125 8GB+256GB SSD Stude...",Blueing,234000.0,3.9,NaN,JUMIA,NaN,laptops,2025-11-13 01:35:13+00:00,web_scrape
1,a261433af780fc99f781522a83458cd5,Macbook PRO Laptop A1278 13.3 Inch Core I5 2.5...,Macbook,187000.0,3.4,NaN,JUMIA,NaN,laptops,2025-11-13 01:35:13+00:00,web_scrape
2,fa51125d5e7a82a495c07eb453169fe8,"Blueing 14"" Laptop N3350 6GB+192GB SSD Portabl...",Blueing,215278.0,4.1,NaN,JUMIA,NaN,laptops,2025-11-13 01:35:13+00:00,web_scrape
3,3c1087bf190638dc1a2637b176fe9c10,Hp EliteBook 840 G7 Intel Core I5 Touchscreen ...,Hp,566000.0,4.0,NaN,JUMIA,NaN,laptops,2025-11-13 01:35:13+00:00,web_scrape
4,936f66875db57fdad5a5674cb5372604,Hp Hp Hp EliteBook 840 G7 10th Gen Intel Core ...,Hp,519990.0,4.1,NaN,JUMIA,NaN,laptops,2025-11-13 01:35:13+00:00,web_scrape


  **FRMAEWORK FOR PHASE 3 - API_AUTH.CSV** 
--------------
#### Phase Overview
In this phase, we ingest the raw Ecommerce Sales dataset and conduct a high-level inspection.
Goals:

- Load the dataset in a reproducible way.

- Examine basic metadata and structural integrity.

- Identify potential data quality issues for later cleaning.

- Document first impressions and possible red flags.

In [ ]:
df_3 = pd.read_csv("../.data_cache/api_auth.csv") 
df_3.head()

In [35]:
df_3.info()

<class 'pandas.DataFrame'>
RangeIndex: 827 entries, 0 to 826
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   City           827 non-null    str    
 1   Country        827 non-null    str    
 2   Latitude       827 non-null    float64
 3   Longitude      827 non-null    float64
 4   Timestamp_UTC  827 non-null    int64  
 5   Temperature_C  827 non-null    float64
 6   Wind_KPH       827 non-null    float64
 7   Condition      827 non-null    str    
 8   AQI_US         817 non-null    float64
 9   CO             817 non-null    float64
 10  NO2            817 non-null    float64
 11  O3             817 non-null    float64
 12  PM2.5          817 non-null    float64
dtypes: float64(9), int64(1), str(3)
memory usage: 84.1 KB


In [ ]:
# Checking for missing values in the dataset to assess data quality and identify any potential issues that may need to be addressed before analysis.
df_3.isna().sum() 

City              0
Country           0
Latitude          0
Longitude         0
Timestamp_UTC     0
Temperature_C     0
Wind_KPH          0
Condition         0
AQI_US           10
CO               10
NO2              10
O3               10
PM2.5            10
dtype: int64

In [ ]:
# Calculating the percentage of missing values in each column to better understand the extent of missing data.  
percent_missing = df_3.isna().sum() * 100 / len(df_3)
print(percent_missing)

City             0.00000
Country          0.00000
Latitude         0.00000
Longitude        0.00000
Timestamp_UTC    0.00000
Temperature_C    0.00000
Wind_KPH         0.00000
Condition        0.00000
AQI_US           1.20919
CO               1.20919
NO2              1.20919
O3               1.20919
PM2.5            1.20919
dtype: float64
